# fixed.xlsx 的数据溯源：1426 条各自来自哪篇文献

`fixed.xlsx` 是 TDT 主表（1426 条 Td）。这个 notebook 把每条数据追溯到它的原始来源，并生成「带来源」的交付表；后面 `source_vs_td.ipynb` 再用它回答"来源和 Td 有没有关系"。

## 要回答的问题
- 每条分子来自 2025 那篇（记 0）、2023 那批（记 1），还是两篇都收录（记 2）？两篇都查不到的记 -1。
- 表里的 `Td(°C)` 抄自哪一边？有没有在清洗过程里被改过？

## 产物
`TDT/out/fixed_去N行_带来源.xlsx`（N = 实际剔除的行数，现在 N = 10），3 个表：

| 表 | 内容 |
|---|---|
| `fixed_去N行带来源` | 剔除 N 行后的主表（现在 N = 10 → 1416 行） |
| `剔除的N行` | 来源不可追溯的行（现在 N = 10） |
| `两篇温度冲突` | 两篇温度不同的 106 个共有分子 |

注：来源与 Td 的相关性、控制组成后的残差检验**都在 `source_vs_td.ipynb`**。

## 匹配原理（关键）
用 **InChIKey** 当匹配键，**不用 SMILES 字符串**：同一分子有多种写法（A 里同时给了带显式 H 的 `smiles` 和规整过的 `canonical_smiles`），字符串直接比会漏（实测字符串比命中 947 条，InChIKey 比命中 966 条）。


## 1. 导入 & 路径

路径全部走仓库根的 `config.py`。**这里没有绝对路径**，换机器不用改 notebook。

In [35]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")  #关掉 RDKit 的警告刷屏

#仓库根：从当前目录往上找带 .git 的目录，这样在仓库里任何位置打开都能 import 到根目录的 config.py
ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import config

DATA = config.DATA_TDT          #仓库里的 TDT/data（fixed.xlsx / unique_random.xlsx / test.xlsx）
SRC = config.SOURCE             #两篇原始文献的数据集（TDT/data/source/）
OUT_DIR = config.OUT            #产物目录，默认 TDT/out；想输出到别处就设环境变量 CHEM_OUT
OUT_DIR.mkdir(parents = True, exist_ok = True)
#产物文件名（"去N行"里的 N 在写盘那一步按实际剔除行数生成，避免名字和数据不一致）

A_FILE = "datasetwithheader.csv"  #2025 那篇的数据集
B_FILE = "Td_zcy.csv"             #2023 那批的数据集
TOL = 0.01                        #温度相等的判定容差（°C）
SEED = 114514                     #固定随机种子

## 2. 读表

`fixed.xlsx` 有两列 `Td(°C)` / `SMILES`。
`load_xlsx` 会先核对列数，列对不上就直接报错，避免后面按列名静静取到错的东西。


In [36]:
def load_xlsx(path, names):
    #读 xlsx 并把列改成 names（上游表的列名不规整，靠位置对齐）
    df = pd.read_excel(path)
    if df.shape[1] != len(names):
        raise ValueError(f"{Path(path).name} 实际 {df.shape[1]} 列，和预期 {len(names)} 列对不上：{list(df.columns)}")
    df.columns = names
    return df


fixed = load_xlsx(DATA / "fixed.xlsx", ["Td(°C)", "SMILES"])
uniq = load_xlsx(DATA / "unique_random.xlsx", ["Td", "smi"])
test = load_xlsx(DATA / "test.xlsx", ["Td", "smi"])

a = pd.read_csv(SRC / A_FILE)   #两篇原始文献的数据集：A 2025、B 2023
b = pd.read_csv(SRC / B_FILE)

print("fixed", fixed.shape, "| A", a.shape, "| B", b.shape)
fixed.head(3)

fixed (1426, 2) | A (1022, 4) | B (886, 2)


,Td(°C),SMILES
0,225.0,O=[N+]([O-])c1cc([N+](=O)[O-])n(-c2nonc2[N+](=...
1,241.0,Nn1cc([N+](=O)[O-])c([N+](=O)[O-])n1
2,136.0,O=[N+]([O-])N1CN=C(C([N+](=O)[O-])([N+](=O)[O-...


## 3. 算 InChIKey

`MolToInchiKey(MolFromSmiles(s))` —— 一个分子一个键，写法不同也算同一个。


In [37]:
def ik(s):
    #SMILES → InChIKey；解析失败返回 None
    m = Chem.MolFromSmiles(str(s))
    return Chem.MolToInchiKey(m) if m else None


fixed["ikey"] = fixed["SMILES"].map(ik)
uniq["ikey"] = uniq["smi"].map(ik)
test["ikey"] = test["smi"].map(ik)
a["ikey"] = a["smiles"].map(ik)
b["ikey"] = b["smiles"].map(ik)

print("无法解析：fixed", int(fixed["ikey"].isna().sum()),
      "| A", int(a["ikey"].isna().sum()), "| B", int(b["ikey"].isna().sum()))

无法解析：fixed 0 | A 0 | B 0


## 4. 标「来源」

`0` 只在 A（2025）、`1` 只在 B（2023）、`2` 两边都有、`-1` 两边都查不到。


In [38]:
ka = set(a["ikey"].dropna())
kb = set(b["ikey"].dropna())

fixed["来源"] = 1
fixed.loc[fixed["ikey"].isin(ka) & ~fixed["ikey"].isin(kb), "来源"] = 0
fixed.loc[fixed["ikey"].isin(ka) & fixed["ikey"].isin(kb), "来源"] = 2
fixed.loc[~fixed["ikey"].isin(ka | kb), "来源"] = -1

fixed["来源"].value_counts().sort_index()

来源
-1     10
 0    560
 1    450
 2    406
Name: count, dtype: int64

## 5. 各来源温度表

`*_by`：每个分子的候选温度列表（同一分子可能有多条记录，比如两篇各给了一个 Td）。


In [39]:
def td_list(df, col):
    #按 InChIKey 把同分子的所有温度收成有序去重列表
    return {k: sorted(set(g[col])) for k, g in df.groupby("ikey")}


A_by = td_list(a, "temp")
B_by = td_list(b, "Td")


pd.DataFrame([
    {"来源": 0, "含义": "只在 2025 论文", "n": int((fixed["来源"] == 0).sum())},
    {"来源": 1, "含义": "只在 2023 来源", "n": int((fixed["来源"] == 1).sum())},
    {"来源": 2, "含义": "两篇都有", "n": int((fixed["来源"] == 2).sum())},
    {"来源": -1, "含义": "两边都查不到", "n": int((fixed["来源"] == -1).sum())},
])

,来源,含义,n
0,0,只在 2025 论文,560
1,1,只在 2023 来源,450
2,2,两篇都有,406
3,-1,两边都查不到,10


## 6. 挑出要剔的行（现在 10 行）

只有一类：**两个来源都查不到**（10 行）——SMILES 修复改动太大，修完的分子在两篇原始表里都找不到同一个 InChIKey。

### 附：原来还剔过 2 行（已修，留个记录）

`fixed` 第 87、434 行原来是 334.1 / 221.0，与两篇来源都对不上。2026-10-24 按方案 B 把它们的 `Td`
改回文献值（**334.1 → 324.5、221.0 → 257.0**），所以现在不再剔除。它们为什么会偏：

根子在 A（2025 那篇）的源数据——**同一化合物录了两条**，其中一条把结构画成了自由基形式（`[N]` / `[NH]`），
两条 Td 不一样：

| 干净的写法 | A 记录 | 自由基写法 | A 记录 |
|---|---|---|---|
| `...c2n[n+]3[n-]c4...`（InChIKey `HNBYBJ…`） | idx=7，**Td=324.5** | `...[N][n+]1...`（`HPFUOJ…`） | idx=8，Td=334.1 |
| `Nc1nnnn1CCn1nc(...)c(Cl)c1[N+](=O)[O-]`（`CPJZNS…`） | idx=668，**Td=257.0** | `[NH]c1nnnn1CCn1...`（`RWISAHV…`） | idx=667，Td=221.0 |

两种写法 InChIKey 不同 → 一路到 `unique_random` 都是两条（89 / 1381、436 / 1322）；
`unique.ipynb` 的 `RULES` 把坏写法修好后两条同键，但 Td 差 **9.6 / 36.0 °C** 超过 `TD_TOL = 5`，没有自动合并；
最后 `DEL_TD` 手工删一条时，留下的正好是**坏写法那条的温度**。**数值本身没被改过**，是"同一条数据录了两遍、留错了一份"。

> 这两行落在 `test.ipynb` 的 `[0:713]` 建模子集里，改回文献值后模型吃到的才是正确标签。

剔除不用绕路：`fixed.xlsx` 第 i 行（0 起）就对应 `Parents/parent_{i:04d}/`，两边都是 1426 个、一一对应。所以要排除谁，直接删对应的 `parent_XXXX` 目录（或从 `BDE_{i:04d}`、`progress.csv` 的同一行去掉）就行，不必再回 `random.xlsx` 找原下标。


In [45]:
def td_matches(tds, td):
    #tds 是该分子的候选温度列表；有没有一个和 td 相等
    return tds is not None and any(abs(v - td) <= TOL for v in tds)


rows = []
for i in range(len(fixed)):
    r = fixed.iloc[i]
    k, td = r["ikey"], r["Td(°C)"]
    av, bv = A_by.get(k), B_by.get(k)
    if av is None and bv is None:
        reason = "来源 = A，修复后 InChIKey 变化导致匹配失败"
    elif not td_matches(av, td) and not td_matches(bv, td):
        #本行 Td 与两篇都对不上：看看是不是"同一分子有两条录入、清洗时留错了那条"
        others = [v for v in (uniq_by.get(k) or []) if abs(v - td) > TOL]
        if others:
            reason = f"同分子重复录入：本行 Td={td}，另一条录入 Td={others[0]}（与来源一致）→ 清洗时留错了那条"
        else:
            reason = "Td 与 A、B 都对不上（来源不明）"
    else:
        continue

    rows.append({
        "原行号(1起)": i + 2,
        "原0起下标": i,
        "要排除的目录": f"parent_{i:04d}",
        "Td(°C)": td,
        "来源": r["来源"],
        "A_2025_Td": av,
        "B_2023_Td": bv,
        "SMILES": r["SMILES"],
        "剔除原因": reason,
    })

removed = pd.DataFrame(rows)
print("要剔", len(removed), "行")
removed[["原行号(1起)", "要排除的目录", "Td(°C)", "来源", "剔除原因"]]

要剔 10 行


,原行号(1起),要排除的目录,Td(°C),来源,剔除原因
0,120,parent_0118,255.53,-1,来源 = A，修复后 InChIKey 变化导致匹配失败
1,171,parent_0169,305.00,-1,来源 = A，修复后 InChIKey 变化导致匹配失败
2,517,parent_0515,261.00,-1,来源 = A，修复后 InChIKey 变化导致匹配失败
3,685,parent_0683,320.00,-1,来源 = A，修复后 InChIKey 变化导致匹配失败
4,695,parent_0693,180.70,-1,来源 = A，修复后 InChIKey 变化导致匹配失败
5,776,parent_0774,231.00,-1,来源 = A，修复后 InChIKey 变化导致匹配失败
6,902,parent_0900,233.00,-1,来源 = A，修复后 InChIKey 变化导致匹配失败
7,974,parent_0972,190.80,-1,来源 = A，修复后 InChIKey 变化导致匹配失败
8,1041,parent_1039,161.00,-1,来源 = A，修复后 InChIKey 变化导致匹配失败
9,1319,parent_1317,159.00,-1,来源 = A，修复后 InChIKey 变化导致匹配失败


## 7. 剔除这些行，生成主表

In [41]:
kept = fixed.drop(index = [x["原0起下标"] for x in removed.to_dict("records")]).copy()

out = kept[["Td(°C)", "SMILES", "来源"]].copy()
out["A_2025_Td"] = kept["ikey"].map(lambda k: A_by.get(k, [None])[0])
out["B_2023_Td"] = kept["ikey"].map(lambda k: B_by.get(k, [None])[0])

da = (kept["Td(°C)"] - out["A_2025_Td"]).abs()
db = (kept["Td(°C)"] - out["B_2023_Td"]).abs()
out["Td与A一致"] = (da <= TOL).where(da.notna(), pd.NA).astype("boolean")
out["Td与B一致"] = (db <= TOL).where(db.notna(), pd.NA).astype("boolean")
out["主表新编号"] = range(len(out))

print("剔", len(removed), "→ 保留", len(out))
out

剔 10 → 保留 1416


,Td(°C),SMILES,来源,A_2025_Td,B_2023_Td,Td与A一致,Td与B一致,主表新编号
0,225.0,O=[N+]([O-])c1cc([N+](=O)[O-])n(-c2nonc2[N+](=...,2,225.0,225.0,True,True,0
1,241.0,Nn1cc([N+](=O)[O-])c([N+](=O)[O-])n1,2,241.0,241.0,True,True,1
2,136.0,O=[N+]([O-])N1CN=C(C([N+](=O)[O-])([N+](=O)[O-...,2,136.0,136.0,True,True,2
3,174.0,[N-]=[N+]=Nc1nc([N+](=O)[O-])n[nH]1,1,NaN,174.0,<NA>,True,3
4,141.0,O=[N+]([O-])c1ncn(NCC([N+](=O)[O-])([N+](=O)[O...,2,141.0,141.0,True,True,4
...,...,...,...,...,...,...,...,...
1421,300.0,O=[N+]([O-])c1n[nH]c([N+](=O)[O-])c1-c1n[nH]c(...,1,NaN,300.0,<NA>,True,1411
1422,182.0,N/C(=N\[N+](=O)[O-])N/N=C/c1c([N+](=O)[O-])cc(...,1,NaN,182.0,<NA>,True,1412
1423,169.0,O=[N+]([O-])/N=c1\[nH]nc2[nH]cnn12,1,NaN,169.0,<NA>,True,1413
1424,237.0,Nn1cnc([N+](=O)[O-])c1,1,NaN,237.0,<NA>,True,1414


## 8. 附表：两篇温度冲突的 106 个分子

两篇都收录、但温度不同的分子，标出 `fixed` 取的是哪一边。

106 条里只有 34 条是真冲突 `|Δ|>5`，其余是 `|Δ|≤5` 的对照行 —— 引用时别把 106 当冲突数。

34 条里面真正脏的 2 条在上面已经修复了，剩下的 32 条不是脏，是「标签噪声」。

这 32 条计划在未来做数据敏感性分析，确认其对模型的影响。

In [42]:
rows = []

for i in range(len(out)):
    k = ik(out.iloc[i]["SMILES"])
    av, bv = A_by.get(k), B_by.get(k)
    if av is None or bv is None:
        continue
    if set(av) == set(bv):
        continue
    td = out.iloc[i]["Td(°C)"]
    picked = "都不是"
    if any(abs(v - td) <= TOL for v in av):
        picked = "A"
    elif any(abs(v - td) <= TOL for v in bv):
        picked = "B"
    rows.append({
        "fixed_Td": td,
        "A_2025_Td": av[0],
        "B_2023_Td": bv[0],
        "A减B": round(av[0] - bv[0], 2),
        "|差|>5": abs(av[0] - bv[0]) > 5,
        "fixed取的是": picked,
        "SMILES": out.iloc[i]["SMILES"],
    })

conflicts = pd.DataFrame(rows)
print("两篇温度不同的共有分子：", len(conflicts), "条，其中 |差|>5：", int(conflicts["|差|>5"].sum()))
conflicts[conflicts["|差|>5"] == True].sort_values("A减B", ascending = False)

两篇温度不同的共有分子： 106 条，其中 |差|>5： 34


,fixed_Td,A_2025_Td,B_2023_Td,A减B,|差|>5,fixed取的是,SMILES
45,335.00,335.00,229.50,105.50,True,A,O=[N+]([O-])c1cc([N+](=O)[O-])c(-c2nnc(-c3nnc(...
25,150.00,254.00,150.00,104.00,True,B,N=C(N)N[N+](=O)[O-]
104,342.00,342.00,242.00,100.00,True,A,Nc1c([N+](=O)[O-])nc([N+](=O)[O-])c(N)[n+]1[O-]
62,295.00,295.00,199.00,96.00,True,A,O=[N+]([O-])c1cc([N+](=O)[O-])[nH]n1
53,285.00,285.00,196.00,89.00,True,A,Cn1nc([N+](=O)[O-])c([N+](=O)[O-])c1[N+](=O)[O-]
58,269.00,269.00,195.00,74.00,True,A,O=[N+]([O-])c1noc(N=C2NCN([N+](=O)[O-])CN2)n1
103,299.00,299.00,255.00,44.00,True,A,O=c1[nH]c2c([N+](=O)[O-])c([N+](=O)[O-])c([N+]...
75,240.00,240.00,202.60,37.40,True,A,O=[N+]([O-])c1cc([N+](=O)[O-])n(/N=N/n2nc([N+]...
96,222.20,222.20,188.00,34.20,True,A,O=[N+]([O-])N1CN([N+](=O)[O-])CC([N+](=O)[O-])...
51,137.00,137.00,111.00,26.00,True,A,O=[N+]([O-])N=C(O)C([N+](=O)[O-])([N+](=O)[O-]...


## 9. 写盘（3 个表）

输出到 `config.OUT`（默认 `TDT/out/`）。表头加粗、冻结首行、开筛选。


In [44]:
#"去N行"里的 N 按实际剔除行数生成，避免文件名/表名和数据对不上
OUT = OUT_DIR / f"fixed_去{len(removed)}行_带来源.xlsx"
SHEET_MAIN = f"fixed_去{len(removed)}行带来源"

with pd.ExcelWriter(OUT, engine = "openpyxl") as w:
    out.to_excel(w, sheet_name = SHEET_MAIN, index = False)
    removed.to_excel(w, sheet_name = f"剔除的{len(removed)}行", index = False)
    conflicts.to_excel(w, sheet_name = "两篇温度冲突", index = False)

from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font

wb = load_workbook(OUT)
for ws in wb.worksheets:
    for c in ws[1]:
        c.font = Font(bold = True)
        c.alignment = Alignment(wrap_text = True, vertical = "center")
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
wb.save(OUT)

print("写出", OUT, "|", OUT.stat().st_size, "字节")
pd.ExcelFile(OUT).sheet_names

写出 C:\Users\czywg\Desktop\Programmer_czy\chem_project\TDT\out\fixed_去10行_带来源.xlsx | 69105 字节


['fixed_去10行带来源', '剔除的10行', '两篇温度冲突']